---

# Practical Examination — CUDA Programming

---

**Program:** B.Tech / M.Tech (Computer Science and Engineering)

**Course Outcome:** CO2 — Apply parallel programming constructs using CUDA

**Difficulty Level:** Easy

**Dataset / Problem:** DS1

**Problem Statement:** Count the number of non-zero elements in a vector using CUDA parallel programming.

---

## 1. Objective

To implement a CUDA kernel that counts the number of non-zero elements in a given integer vector using parallel reduction on the GPU, and to verify the result against a sequential CPU-based computation.

## 2. Theory

### 2.1 CUDA Parallel Reduction

Reduction is the process of combining all elements of an array into a single scalar value using an associative operator (e.g., sum, max, count). In a sequential approach, this requires O(N) time. CUDA enables parallel reduction by:

1. Dividing the input vector across multiple threads.
2. Each thread evaluates whether its assigned element is non-zero and contributes 1 or 0 to a local count.
3. An atomic operation (`atomicAdd`) is used to safely accumulate partial results from all threads into a global counter without race conditions.

### 2.2 Atomic Operations

`atomicAdd(address, val)` reads the value at `address`, adds `val` to it, and writes the result back — all as a single, indivisible operation. This prevents concurrent threads from corrupting the shared counter.

### 2.3 Thread Indexing

Each thread computes its global index as:

```
idx = blockIdx.x * blockDim.x + threadIdx.x
```

A boundary check (`idx < N`) ensures threads beyond the array length do not access invalid memory.

## 3. Environment Setup

In [1]:
# Verify GPU availability in the Colab runtime
# Runtime > Change runtime type > Hardware accelerator > GPU (T4)
!nvidia-smi

Mon Jun  1 05:04:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Install PyCUDA for writing and executing CUDA kernels from Python
!pip install pycuda --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 3.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 10.6 MB/s eta 0:00:00


## 4. Implementation

In [3]:
import pycuda.autoinit                  # Automatically initializes the CUDA context
import pycuda.driver as cuda
from pycuda.compiler import SourceModule
import numpy as np
import time

In [4]:
# -----------------------------------------------------------------------
# CUDA Kernel Definition
# -----------------------------------------------------------------------
# Each thread checks one element of the input vector.
# If the element is non-zero, atomicAdd increments the global counter.
# -----------------------------------------------------------------------

cuda_code = """
__global__ void count_nonzero(int *d_vec, int *d_count, int N)
{
    // Compute the global thread index
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    // Boundary check: only process valid indices
    if (idx < N)
    {
        if (d_vec[idx] != 0)
        {
            // Atomically increment the counter to prevent race conditions
            atomicAdd(d_count, 1);
        }
    }
}
"""

# Compile the CUDA kernel at runtime
module = SourceModule(cuda_code)
count_nonzero = module.get_function("count_nonzero")

In [5]:
# -----------------------------------------------------------------------
# Input Data Preparation
# -----------------------------------------------------------------------

N = 1_000_000  # Vector size: 1 million elements

np.random.seed(42)
# Generate random integers in range [-5, 5]; approximately 2/11 will be zero
h_vec = np.random.randint(-5, 6, size=N).astype(np.int32)

print(f"Vector size          : {N:,}")
print(f"Sample (first 10)    : {h_vec[:10]}")
print(f"Non-zero count (CPU) : {np.count_nonzero(h_vec):,}  [reference value]")

Vector size          : 1,000,000
Sample (first 10)    : [ 1 -2  5  2 -1  1  4 -3  1  5]
Non-zero count (CPU) : 909,091  [reference value]


In [6]:
# -----------------------------------------------------------------------
# GPU Memory Allocation and Data Transfer (Host -> Device)
# -----------------------------------------------------------------------

# Allocate and copy the input vector to GPU memory
d_vec = cuda.mem_alloc(h_vec.nbytes)
cuda.memcpy_htod(d_vec, h_vec)

# Allocate the counter on GPU and initialize to 0
h_count = np.zeros(1, dtype=np.int32)
d_count = cuda.mem_alloc(h_count.nbytes)
cuda.memcpy_htod(d_count, h_count)

In [7]:
# -----------------------------------------------------------------------
# Kernel Launch Configuration
# -----------------------------------------------------------------------

THREADS_PER_BLOCK = 256
# Number of blocks needed to cover all N elements
BLOCKS = (N + THREADS_PER_BLOCK - 1) // THREADS_PER_BLOCK

print(f"Threads per block    : {THREADS_PER_BLOCK}")
print(f"Number of blocks     : {BLOCKS}")
print(f"Total threads        : {THREADS_PER_BLOCK * BLOCKS:,}")

Threads per block    : 256
Number of blocks     : 3907
Total threads        : 1,000,192


In [8]:
# -----------------------------------------------------------------------
# Kernel Execution
# -----------------------------------------------------------------------

start = time.time()

count_nonzero(
    d_vec,
    d_count,
    np.int32(N),
    block=(THREADS_PER_BLOCK, 1, 1),
    grid=(BLOCKS, 1)
)

# Synchronize to ensure kernel completion before reading results
cuda.Context.synchronize()
gpu_time = time.time() - start

# Copy result from Device -> Host
cuda.memcpy_dtoh(h_count, d_count)

print(f"GPU Execution Time   : {gpu_time * 1000:.4f} ms")

GPU Execution Time   : 2.4679 ms


## 5. Results and Verification

In [9]:
# Sequential CPU computation for correctness verification
start_cpu = time.time()
cpu_count = int(np.count_nonzero(h_vec))
cpu_time = time.time() - start_cpu

gpu_count = int(h_count[0])

print("=" * 45)
print("         RESULTS SUMMARY")
print("=" * 45)
print(f"  Vector size          : {N:,}")
print(f"  GPU count (CUDA)     : {gpu_count:,}")
print(f"  CPU count (NumPy)    : {cpu_count:,}")
print(f"  Match                : {gpu_count == cpu_count}")
print("-" * 45)
print(f"  GPU time             : {gpu_time * 1000:.4f} ms")
print(f"  CPU time             : {cpu_time * 1000:.4f} ms")
print("=" * 45)

         RESULTS SUMMARY
  Vector size          : 1,000,000
  GPU count (CUDA)     : 909,091
  CPU count (NumPy)    : 909,091
  Match                : True
---------------------------------------------
  GPU time             : 2.4679 ms
  CPU time             : 0.6430 ms


## 6. Conclusion

A CUDA kernel was successfully implemented to count non-zero elements in a vector of one million integers. Each GPU thread independently evaluated one element and used `atomicAdd` to safely update the shared counter. The GPU result matched the CPU reference exactly, confirming correctness. The parallel execution across thousands of threads demonstrates the efficiency of CUDA for element-wise vector operations.

However, the GPU execution time (2.47 ms) was higher than the CPU time (0.64 ms). This is expected for this problem: counting non-zero elements is a memory-bound operation with minimal compute per element, and the overhead of kernel launch, thread scheduling, and atomic contention on a single counter outweighs the benefit of parallelism at this scale. CUDA's advantage becomes significant for larger vectors, compute-intensive kernels, or problems where work is distributed across multiple independent accumulators rather than a single atomic counter.

---

*End of Practical — DS1*

---